# Final ML experiment: random oversampling

Compare the current weighted baseline (`scale_pos_weight=7`) with RandomOverSampler ratios 0.25, 0.50, and 1.00 (`scale_pos_weight=1`). A ratio means minority / majority after resampling. Only training data is resampled by duplicating minority examples with replacement; validation remains unchanged and test arrays are never loaded. No SMOTE is used.

Freeze the latest selected top-25 configuration from `selected_configuration.json`, including its effective number of trees (445 at creation). Do not run early stopping or hyperparameter tuning again. Fit feature selection on the complete training split before sampling so all candidates use the same 25 columns.

PR-AUC for ranking means sklearn **average precision**, matching prior experiments. Also report trapezoidal PR area separately. Select each model's threshold by validation F1 over all distinct validation probability thresholds, breaking ties by recall then precision. Threshold tuning and evaluation share validation data, so these are selection scores rather than an unbiased final performance estimate. The existing test set is left untouched.


In [1]:
from pathlib import Path
from time import perf_counter
from importlib.metadata import version
from uuid import uuid4
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from IPython.display import display
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score, precision_recall_curve,
    auc, confusion_matrix)
from xgboost import XGBClassifier

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/processed_datasets.npz").exists())
DATA_PATH = ROOT / "data/processed_datasets.npz"
CONFIG_PATH = ROOT / "data/xgboost_weighted_improvement/selected_configuration.json"
selected_config = json.loads(CONFIG_PATH.read_text())
config = selected_config["configuration_and_metrics"]
N_FEATURES = int(config["n_features"])
assert N_FEATURES == 25 and config["scale_pos_weight"] == 7
MODEL_PARAMS = {key: config[key] for key in [
    "learning_rate", "max_depth", "subsample", "colsample_bytree",
    "min_child_weight", "random_state", "n_jobs", "eval_metric"]}
MODEL_PARAMS["n_estimators"] = int(config.get("effective_trees", config["n_estimators"]))
CANDIDATES = [("weighted_baseline", None, 7)] + [
    (f"oversampling_{ratio:.2f}", ratio, 1) for ratio in [0.25, 0.50, 1.00]]
SEARCH_ID = uuid4().hex
OUTPUT_DIR = ROOT / "data/xgboost_oversampling" / SEARCH_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with np.load(DATA_PATH, allow_pickle=False) as data:
    X_train, y_train = data["X_train"], data["Y_train"]
    X_val, y_val = data["X_val"], data["Y_val"]
assert len(X_train) == len(y_train) and len(X_val) == len(y_val)
assert X_train.shape[1] == X_val.shape[1] >= N_FEATURES
assert set(np.unique(y_train)) == set(np.unique(y_val)) == {0, 1}
assert np.sum(y_train == 0) > np.sum(y_train == 1)

def array_hash(*arrays):
    digest = hashlib.sha256()
    for array in arrays:
        digest.update(str((array.shape, str(array.dtype))).encode())
        digest.update(np.ascontiguousarray(array).tobytes())
    return digest.hexdigest()

train_hash = array_hash(X_train, y_train)
val_hash = array_hash(X_val, y_val)
DB_PATH = ROOT / "experiments/mlflow_churn.db"
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")
EXPERIMENT_NAME = "churn-xgboost-final-random-oversampling"
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    mlflow.create_experiment(EXPERIMENT_NAME,
        artifact_location=(ROOT / "experiments/mlruns/final_random_oversampling").as_uri())
mlflow.set_experiment(EXPERIMENT_NAME)
print("Frozen parameters:", MODEL_PARAMS)
print("Train:", X_train.shape, "Validation:", X_val.shape)
print("Output:", OUTPUT_DIR)


Frozen parameters: {'learning_rate': 0.03, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1, 'random_state': 42, 'n_jobs': 8, 'eval_metric': 'aucpr', 'n_estimators': 445}
Train: (659028, 44) Validation: (82379, 44)
Output: /home/rami/Desktop/Projects/mlops/data/xgboost_oversampling/4d6849cd98364ea7baaae4bb310edf40


In [2]:
def evaluate(y, probabilities, threshold):
    predictions = probabilities >= threshold
    precision, recall, _ = precision_recall_curve(y, probabilities)
    ap = average_precision_score(y, probabilities)
    return {"pr_auc": ap, "average_precision": ap,
            "pr_auc_trapezoidal": auc(recall, precision),
            "roc_auc": roc_auc_score(y, probabilities),
            "f1": f1_score(y, predictions, zero_division=0),
            "precision": precision_score(y, predictions, zero_division=0),
            "recall": recall_score(y, predictions, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y, predictions),
            "accuracy": accuracy_score(y, predictions)}

def choose_threshold(y, probabilities):
    precision, recall, thresholds = precision_recall_curve(y, probabilities)
    precision, recall = precision[:-1], recall[:-1]
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros_like(precision), where=(precision + recall) > 0)
    table = pd.DataFrame({"threshold": thresholds, "f1": f1,
                          "precision": precision, "recall": recall})
    table = table.sort_values(["f1", "recall", "precision", "threshold"],
                             ascending=[False, False, False, True])
    return float(table.iloc[0].threshold), table

results = []
reference_features = None
for name, ratio, weight in CANDIDATES:
    print("Training", name, flush=True)
    run_dir = OUTPUT_DIR / name
    run_dir.mkdir(exist_ok=True)
    sampler = "passthrough" if ratio is None else RandomOverSampler(
        sampling_strategy=ratio, random_state=MODEL_PARAMS["random_state"], shrinkage=None)
    pipeline = Pipeline([
        ("feature_selection", SelectKBest(f_classif, k=N_FEATURES)),
        ("oversampling", sampler),
        ("model", XGBClassifier(**MODEL_PARAMS, scale_pos_weight=weight)),
    ])
    with mlflow.start_run(run_name=name, tags={
        "search_id": SEARCH_ID, "stage": "final_ml_experiment",
        "evaluation_split": "validation", "pr_auc_definition": "average_precision",
        "baseline_source_run_id": config.get("run_id", "unknown"),
    }) as run:
        mlflow.log_params({**MODEL_PARAMS, "n_features": N_FEATURES,
            "scale_pos_weight": weight, "sampling_strategy": "none" if ratio is None else ratio,
            "sampler": "none" if ratio is None else "RandomOverSampler",
            "shrinkage": "none", "train_rows_original": len(y_train), "validation_rows": len(y_val),
            "train_sha256": train_hash, "validation_sha256": val_hash,
            "threshold_selection": "max_validation_f1_then_recall_then_precision"})
        start = perf_counter()
        pipeline.fit(X_train, y_train)
        fit_seconds = perf_counter() - start
        indices = pipeline.named_steps["feature_selection"].get_support(indices=True)
        if reference_features is None:
            reference_features = indices
        np.testing.assert_array_equal(indices, reference_features)
        sample_indices = (np.arange(len(y_train)) if ratio is None else sampler.sample_indices_)
        counts = np.bincount(y_train[sample_indices], minlength=2)
        assert counts[0] == np.sum(y_train == 0)
        # Original examples are retained; only minority examples are duplicated.
        np.testing.assert_array_equal(sample_indices[:len(y_train)], np.arange(len(y_train)))
        if ratio is not None:
            assert counts[1] == int(counts[0] * ratio)
            assert np.all(y_train[sample_indices[len(y_train):]] == 1)
        else:
            assert counts[1] == np.sum(y_train == 1)
        scores = pipeline.predict_proba(X_val)[:, 1]
        assert len(scores) == len(y_val)
        threshold, threshold_table = choose_threshold(y_val, scores)
        scores_metrics = evaluate(y_val, scores, threshold)
        np.testing.assert_allclose(scores_metrics["f1"], threshold_table.iloc[0].f1)
        # Check that the fixed-tree baseline reproduces the existing selected model.
        if ratio is None:
            np.testing.assert_allclose(average_precision_score(y_val, scores),
                                       config["val_average_precision"], rtol=0, atol=1e-10)
            mlflow.log_metrics({"reference_threshold_" + k: v for k, v in
                evaluate(y_val, scores, selected_config["decision_threshold"]).items()})
            mlflow.log_param("reference_threshold", selected_config["decision_threshold"])
        mlflow.log_params({"decision_threshold": threshold,
            "train_rows_resampled": int(counts.sum()), "train_non_churn": int(counts[0]),
            "train_churn": int(counts[1])})
        mlflow.log_metrics({**scores_metrics, "fit_seconds": fit_seconds})
        threshold_table.to_csv(run_dir / "validation_thresholds.csv", index=False)
        np.savez_compressed(run_dir / "validation_predictions.npz", y=y_val, probabilities=scores)
        np.save(run_dir / "training_sample_indices.npy", sample_indices)
        pd.DataFrame(confusion_matrix(y_val, scores >= threshold, labels=[0, 1]),
            index=["actual_no_churn", "actual_churn"],
            columns=["predicted_no_churn", "predicted_churn"]).to_csv(run_dir / "confusion_matrix.csv")
        mlflow.log_artifacts(str(run_dir), artifact_path="evaluation")
        mlflow.log_dict({"indices": indices.tolist()}, "selected_features.json")
        mlflow.log_dict({"python": platform.python_version(), **{pkg: version(pkg) for pkg in
            ["numpy", "pandas", "scikit-learn", "imbalanced-learn", "xgboost", "mlflow"]}}, "environment.json")
        mlflow.log_dict(selected_config, "baseline_configuration.json")
        mlflow.log_dict({"decision_threshold": threshold,
            "prediction_rule": "predict_proba(X)[:, 1] >= decision_threshold",
            "note": "Pipeline.predict uses 0.5; sampling is skipped at inference."}, "prediction_settings.json")
        model_info = mlflow.sklearn.log_model(pipeline, name="model", serialization_format="cloudpickle")
        # Validate the stored pipeline, including sampler behavior at inference.
        restored = mlflow.sklearn.load_model(model_info.model_uri)
        np.testing.assert_allclose(restored.predict_proba(X_val[:128])[:, 1], scores[:128])
        results.append({"candidate": name, "sampling_ratio": ratio, "scale_pos_weight": weight,
            "threshold": threshold, **scores_metrics, "fit_seconds": fit_seconds,
            "train_non_churn": int(counts[0]), "train_churn": int(counts[1]),
            "run_id": run.info.run_id, "model_uri": model_info.model_uri})
        pd.DataFrame(results).to_csv(OUTPUT_DIR / "comparison.csv", index=False)
    print(f"{name}: PR-AUC={scores_metrics['pr_auc']:.6f}, F1={scores_metrics['f1']:.6f}, "
          f"threshold={threshold:.6f}, fit={fit_seconds:.2f}s", flush=True)
# Prove that neither training input nor validation input was modified in place.
assert array_hash(X_train, y_train) == train_hash
assert array_hash(X_val, y_val) == val_hash


Training weighted_baseline
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/30 11:02:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
weighted_baseline: PR-AUC=0.206725, F1=0.269538, threshold=0.511252, fit=14.73s
Training oversampling_0.25
/home/rami/miniconda3/envs/mlops/lib/p

In [3]:
comparison = pd.DataFrame(results).sort_values(["pr_auc", "f1"], ascending=False).reset_index(drop=True)
baseline = comparison.loc[comparison.candidate == "weighted_baseline"].iloc[0]
for metric in ["pr_auc", "roc_auc", "f1", "precision", "recall", "balanced_accuracy", "accuracy"]:
    comparison[metric + "_delta_vs_baseline"] = comparison[metric] - baseline[metric]
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
ap_winner = comparison.iloc[0]
f1_winner = comparison.sort_values(["f1", "pr_auc"], ascending=False).iloc[0]
summary = {
    "search_id": SEARCH_ID, "validation_pr_auc_definition": "average_precision",
    "best_validation_pr_auc": {key: ap_winner[key] for key in ["candidate", "pr_auc", "f1", "threshold", "run_id"]},
    "best_validation_f1": {key: f1_winner[key] for key in ["candidate", "pr_auc", "f1", "threshold", "run_id"]},
    "test_evaluated": False,
    "fit_seconds_definition": "feature selection + training resampling + XGBoost fitting; excludes prediction and logging",
    "next_phase": "MLOps",
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
with mlflow.start_run(run_name="final_oversampling_comparison", tags={"search_id": SEARCH_ID, "role": "comparison"}):
    mlflow.log_artifact(str(OUTPUT_DIR / "comparison.csv"))
    mlflow.log_dict(summary, "summary.json")
    mlflow.log_params({"pr_auc_winner": ap_winner.candidate, "f1_winner": f1_winner.candidate})
columns = ["candidate", "threshold", "pr_auc", "roc_auc", "f1", "precision", "recall",
           "balanced_accuracy", "accuracy", "fit_seconds", "train_non_churn", "train_churn"]
print(comparison[columns].to_string(index=False))
print(json.dumps(summary, indent=2))
display(comparison[columns])


        candidate  threshold   pr_auc  roc_auc       f1  precision   recall  balanced_accuracy  accuracy  fit_seconds  train_non_churn  train_churn
weighted_baseline   0.511252 0.206725 0.684118 0.269538   0.190537 0.460455           0.622575  0.752546    14.726971           593685        65343
oversampling_0.50   0.415116 0.206334 0.683866 0.269085   0.195714 0.430460           0.617879  0.768133    21.479695           593685       296842
oversampling_1.00   0.583246 0.206204 0.683880 0.269721   0.194390 0.440377           0.619752  0.763556    27.457909           593685       593685
oversampling_0.25   0.256456 0.206180 0.684015 0.268878   0.192334 0.446621           0.620098  0.759174    18.099087           593685       148421
{
  "search_id": "4d6849cd98364ea7baaae4bb310edf40",
  "validation_pr_auc_definition": "average_precision",
  "best_validation_pr_auc": {
    "candidate": "weighted_baseline",
    "pr_auc": 0.20672509732768157,
    "f1": 0.26953810871824274,
    "threshold": 0